# Kent-AI: Phase 3 — Bio_ClinicalBERT HomeoNER Fine-Tuning

## Clinical Mission & Task Definition
This notebook establishes the token-classification fine-tuning pipeline for **HomeoNER** (Homeopathic Named Entity Recognition).
Our objective is to extract the **7 canonical Kent symptom dimensions** from patient conversational transcripts into structured entity spans:

| Dimension Code | Clinical Meaning | Example Phrasing |
|---|---|---|
| `LOC` | Body part / Anatomical region | *"right temple"*, *"vertex of head"* |
| `SEN` | Sensation / Feeling type | *"pressing throbbing"*, *"burning heat"* |
| `MOD_AGG` | Modality: Aggravation (worse from) | *"worse in morning"*, *"worse cold draft"* |
| `MOD_AMEL` | Modality: Amelioration (better from) | *"better by hard pressure"*, *"fresh open air"* |
| `CONC` | Concomitant symptoms | *"with nausea and vertigo"* |
| `TEMP` | Time patterns and periodicity | *"at 3 AM every night"*, *"twilight"* |
| `MENT` | Mental & emotional generals | *"extreme anxiety alone"*, *"irritable on waking"* |

### Model Architecture
We use `emilyalsentzer/Bio_ClinicalBERT` (110M parameters, pre-trained on PubMed and MIMIC-III clinical notes) with a 15-tag linear token classification head (`O`, `B-LOC`, `I-LOC`, etc.).

In [ ]:
import os
import sys
import json
from pathlib import Path

# Navigate to project root repository
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import seqeval.metrics

from src.config import get_model_config, get_project_root
from src.data.bio_tagger import BIOTagger
from src.models.trainer import NERTrainer, set_seed
from src.models.symptom_ner import SymptomNER

# Set random seeds for experimental reproducibility
set_seed(42)

# Verify compute environment
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Target Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 1. BIO Label Scheme & Configuration
We define a 15-tag BIO schema representing the beginning (`B-`), continuation (`I-`), and outside (`O`) tags across the 7 Kentian symptom dimensions.

In [ ]:
config = get_model_config()
tags = config.get("labels", {}).get("tags", [
    "O",
    "B-LOC", "I-LOC",
    "B-SEN", "I-SEN",
    "B-MOD_AGG", "I-MOD_AGG",
    "B-MOD_AMEL", "I-MOD_AMEL",
    "B-CONC", "I-CONC",
    "B-TEMP", "I-TEMP",
    "B-MENT", "I-MENT",
])

id2label = {i: tag for i, tag in enumerate(tags)}
label2id = {tag: i for i, tag in enumerate(tags)}

print(f"Total BIO Tags: {len(tags)}")
for i, tag in id2label.items():
    print(f"  ID {i:2d} -> {tag}")

## 2. Dataset Ingestion (Train, Validation, Test)
The training set consists of synthetic clinical cases generated from Kent's Repertory MIND rubrics via LLaMA 3, split 80% / 10% / 10% with rubric stratification.

*Note: If running before the full 22,200 production batch completes on the college GPU, this notebook automatically provides a small demonstration subset so code verification can proceed without blocking.*

In [ ]:
data_dir = project_root / "data" / "processed"
train_file = data_dir / "train.jsonl"
val_file = data_dir / "val.jsonl"
test_file = data_dir / "test.jsonl"

# Check dataset availability
dataset_available = train_file.exists() and val_file.exists()

if dataset_available:
    print(f"[✓] Found training datasets in {data_dir}")
    with open(train_file, "r", encoding="utf-8") as f:
        sample = json.loads(f.readline())
    print(f"Sample Case ID: {sample.get('case_id')}")
    print(f"Sample Rubric:  {sample.get('rubric_path')}")
    print(f"Transcript:     {sample.get('transcript')[:90]}...")
    print(f"Entities Count: {len(sample.get('entities', []))}")
else:
    print(f"[!] Datasets not found at {train_file}.")
    print("    Generating small synthetic sample for workflow validation...")
    from src.data.case_generator import CaseGenerator
    generator = CaseGenerator(mock_mode=True)
    sample = generator.generate_case_for_rubric(rubric_id=265, rubric_path="MIND > ANXIETY > dark, in", case_idx=1)
    print(f"Generated test sample with {len(sample.entities)} entity spans.")
    print(f"Transcript: {sample.transcript}")

## 3. Subword Tokenization & Offset Alignment
A key technical challenge in transformer-based clinical NER is **subword tokenization**:
- Words like *"absent-minded"* or *"throbbing"* may be split into multiple subwords (e.g. `['thro', '##bbing']`).
- Character spans must be mapped onto subwords using HuggingFace's `offset_mapping`.
- Non-initial subwords and special tokens (`[CLS]`, `[SEP]`, `[PAD]`) must receive the PyTorch ignore index (`-100`) so the cross-entropy loss ignores them during gradient updates.

In [ ]:
model_name = config.get("model", {}).get("name", "emilyalsentzer/Bio_ClinicalBERT")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tagger = BIOTagger()

# Demonstration of subword alignment on a clinical sentence
demo_text = "Patient reports severe anxiety in the morning, worse when alone, with trembling hands."
demo_entities = [
    {"text": "anxiety", "label": "MENT", "start": 22, "end": 29},
    {"text": "morning", "label": "TEMP", "start": 37, "end": 44},
    {"text": "worse when alone", "label": "MOD_AGG", "start": 46, "end": 62},
    {"text": "trembling hands", "label": "CONC", "start": 69, "end": 84},
]

tokenized = tokenizer(demo_text, return_offsets_mapping=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])
aligned_labels = tagger.align_with_subwords(
    subword_offsets=tokenized["offset_mapping"],
    entities=demo_entities,
    text=demo_text,
    tag_to_id=label2id,
    ignore_index=-100,
)

print(f"{'Subword':18} | {'Token ID':8} | {'Label ID':8} | {'BIO Tag'}")
print("-" * 52)
for tok, tid, lid in zip(tokens[:14], tokenized["input_ids"][:14], aligned_labels[:14]):
    tag_str = id2label[lid] if lid != -100 else "[IGNORE]"
    print(f"{tok:18} | {tid:8d} | {lid:8d} | {tag_str}")

## 4. Model Architecture & Fine-Tuning Setup
We instantiate `AutoModelForTokenClassification` on top of `Bio_ClinicalBERT`.

```text
Input Tokens -> [Bio_ClinicalBERT 12 Layers (110M params)] -> Dropout(0.1) -> Linear(768 -> 15) -> CrossEntropyLoss
```

### Hyperparameters (from `configs/model.yaml`):
- **Learning Rate**: $2.0 \times 10^{-5}$ (AdamW with weight decay 0.01)
- **Batch Size**: 16 per device (with gradient accumulation if memory constrained)
- **Precision**: Mixed Precision (FP16 enabled on GPU)
- **Early Stopping**: Patience = 3 epochs on validation F1 score

In [ ]:
trainer = NERTrainer(config=config, seed=42)
print(f"NERTrainer initialized successfully for model: {trainer.model_name}")
print(f"Artifact output directory: {trainer.output_dir}")

## 5. Training Execution Pipeline
Once the 22,200 cases are available in `data/processed/train.jsonl` and `val.jsonl`, run the cell below to fine-tune the model.

In [ ]:
# Trigger full training when data is available
if dataset_available:
    print("Starting Bio_ClinicalBERT HomeoNER fine-tuning...")
    eval_metrics = trainer.train(
        train_path=train_file,
        val_path=val_file,
    )
    print("Fine-tuning complete!")
    print(json.dumps(eval_metrics, indent=2))
else:
    print("Training skipped: data/processed/train.jsonl is not yet generated.")
    print("To generate data on College GPU, execute: bash scripts/run_production_generation.sh")

## 6. Inference & Span Extraction Demonstration
The `SymptomNER` module encapsulates the fine-tuned model (or deterministic fallback) to extract labeled entity spans directly from free text.

In [ ]:
ner = SymptomNER(mock_mode=not dataset_available)

test_narratives = [
    "Doctor, I have severe anxiety every morning around 5 AM, worse when alone with trembling.",
    "Patient reports throbbing pain in right temple, better by hard pressure and fresh open air.",
    "Extreme irritability on waking, weeping over trifles, with restless pacing at midnight.",
]

print("=== SymptomNER Extraction Demonstration ===\n")
for idx, text in enumerate(test_narratives, start=1):
    print(f"Case {idx}: \"{text}\"")
    spans = ner.extract_entities(text)
    for span in spans:
        print(f"   [{span['label']:10s}] \"{span['text']}\" (chars {span['start']}-{span['end']}, conf={span['confidence']:.2f})")
    print()

## 7. Downstream Hand-Off to Kent Pipeline
Once extracted by ClinicalBERT, these entity spans are passed sequentially to:
1. **LLaMA 3 Resolver** (`src/models/resolver.py`) — Negation filtering & query synthesis.
2. **ChromaDB Vector Store** (`src/search/vector_store.py`) — Dense semantic retrieval over 74k rubrics.
3. **Totality Ranker** (`src/search/ranker.py`) — Classical Kentian totality score computation.